# Training + Results for the Model

## Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import pymc as pm
import arviz as az
import pytensor.tensor as pt

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


## Loading Data

In [2]:
DATA_PATH = Path("../final_data")
df = pd.read_csv(DATA_PATH / "model_ready_dataset.csv")
df.head()

,date,topic_Economy_taxes_tariffs_seed_binary,topic_Border_security_immigration_seed_binary,topic_Foreign_policy_seed_binary,topic_Energy_seed_binary,topic_Environment_seed_binary,topic_Government_reform_seed_binary,topic_Legal_battles_seed_binary,topic_Media_attacks_seed_binary,topic_Political_opposition_attacks_seed_binary,...,vix_close,delta_vix,vix_pct_change,vix_lag1,delta_vix_lag1,sp500_close,sp500_return,sp500_return_lag1,sp500_rolling_vol_5d,sp500_rolling_vol_10d
0,2025-05-27,1,1,1,0,1,1,1,1,1,...,18.96,-1.61,-0.078269,20.57,-1.72,5921.54,0.020459,-0.006708,0.013511,0.009668
1,2025-05-28,1,1,1,1,0,1,1,1,1,...,19.31,0.35,0.018460,18.96,-1.61,5888.55,-0.005571,0.020459,0.013611,0.009650
2,2025-05-30,1,1,1,1,0,1,0,1,1,...,18.57,-0.61,-0.031804,19.18,-0.13,5911.69,-0.000081,0.004011,0.010970,0.009639
3,2025-06-04,0,1,1,0,0,0,0,0,1,...,17.61,-0.08,-0.004522,17.69,-0.67,5970.81,0.000074,0.005800,0.002641,0.009538
4,2025-06-05,1,1,1,0,1,1,1,1,1,...,18.48,0.87,0.049404,17.61,-0.08,5939.30,-0.005277,0.000074,0.004303,0.007905


### Detailed Column Descriptions

In [4]:
summary = df.describe()
for col in summary.columns:
    print("{\n\tColumn: " + col + ",")
    print("\t" + str(df[col].dtype) + ",")
    print("\t" + str(summary[col][["mean","std","min","max"]]))
    print("}")

{
	Column: topic_Economy_taxes_tariffs_seed_binary,
	int64,
	mean    0.945355
std     0.227909
min     0.000000
max     1.000000
Name: topic_Economy_taxes_tariffs_seed_binary, dtype: float64
}
{
	Column: topic_Border_security_immigration_seed_binary,
	int64,
	mean    0.890710
std     0.312858
min     0.000000
max     1.000000
Name: topic_Border_security_immigration_seed_binary, dtype: float64
}
{
	Column: topic_Foreign_policy_seed_binary,
	int64,
	mean    0.972678
std     0.163468
min     0.000000
max     1.000000
Name: topic_Foreign_policy_seed_binary, dtype: float64
}
{
	Column: topic_Energy_seed_binary,
	int64,
	mean    0.814208
std     0.390006
min     0.000000
max     1.000000
Name: topic_Energy_seed_binary, dtype: float64
}
{
	Column: topic_Environment_seed_binary,
	int64,
	mean    0.606557
std     0.489854
min     0.000000
max     1.000000
Name: topic_Environment_seed_binary, dtype: float64
}
{
	Column: topic_Government_reform_seed_binary,
	int64,
	mean    0.874317
std     0.332

### Feature Setup and Selection

In [3]:
modeling_df = df.copy()

modeling_df["date"] = pd.to_datetime(modeling_df["date"])
modeling_df = modeling_df.sort_values("date").reset_index(drop=True)
modeling_df["abs_delta_vix"] = modeling_df["delta_vix"].apply(np.abs)

# response (Emission) variable
# we use absolute value to get the intensity instead of direction
y_col = "abs_delta_vix"

# nl features
seed_count_cols = [c for c in modeling_df.columns if c.startswith("topic_") and c.endswith("_seed_count")]
seed_binary_cols = [c for c in modeling_df.columns if c.startswith("topic_") and c.endswith("_seed_binary")]
sentence_count_cols = [c for c in modeling_df.columns if c.endswith("_topic_sentence_count")]
nmf_score_cols = [c for c in modeling_df.columns if c.startswith("nmf_topic_") and c.endswith("_score")]
nmf_binary_cols = [c for c in modeling_df.columns if c.startswith("nmf_topic_") and c.endswith("_binary")]
embedding_binary_cols = [c for c in modeling_df.columns if c.startswith("embedding_topic_") and c.endswith("_binary")]

financial_control_cols = [
    "vix_lag1",
    "delta_vix_lag1",
    "sp500_return",
    "sp500_return_lag1",
    "sp500_rolling_vol_5d",
    "sp500_rolling_vol_10d",
]

financial_control_cols = [c for c in financial_control_cols if c in modeling_df.columns]

In [4]:
# using nmf scores because they sum to 1 in each row. Using probabilities is better than raw counts or binary
nlp_cols = nmf_score_cols
# for the nlp-only model
X_nlp_cols = nlp_cols
# for the robustness check model
X_full_cols = X_nlp_cols + financial_control_cols

print("NLP columns:", X_nlp_cols)
print("Financial controls:", financial_control_cols)

NLP columns: ['nmf_topic_0_score', 'nmf_topic_1_score', 'nmf_topic_2_score', 'nmf_topic_3_score', 'nmf_topic_4_score', 'nmf_topic_5_score', 'nmf_topic_6_score', 'nmf_topic_7_score', 'nmf_topic_8_score', 'nmf_topic_9_score', 'nmf_topic_10_score']
Financial controls: ['vix_lag1', 'delta_vix_lag1', 'sp500_return', 'sp500_return_lag1', 'sp500_rolling_vol_5d', 'sp500_rolling_vol_10d']


In [5]:
def make_model_arrays(data, y_col, x_cols):
    keep_cols = [y_col] + x_cols
    d = data[keep_cols].copy()
    d = d.replace([np.inf, -np.inf], np.nan)
    d = d.dropna(subset=[y_col]).reset_index(drop=True)

    y = d[y_col].to_numpy().astype(float)
    X_raw = d[x_cols].to_numpy().astype(float)

    # impute nans with the mean then z-scale
    x_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    X = x_pipe.fit_transform(X_raw)

    return d, y, X, x_pipe

d_nlp, y, X_nlp, x_pipe_nlp = make_model_arrays(modeling_df, y_col, X_nlp_cols)
d_full, y_full, X_full, x_pipe_full = make_model_arrays(modeling_df, y_col, X_full_cols)

print(y.shape, X_nlp.shape)
print(y_full.shape, X_full.shape)

(183,) (183, 11)
(183,) (183, 17)


## Modeling

### Let HMM infer states
Below is just initialization of the HMM using the observed data to infer what the hidden regimes are given there are 3 states and the observed changes in VIX.

In [6]:
from hmmlearn.hmm import GaussianHMM

K = 3

hmm = GaussianHMM(
    n_components=K,
    covariance_type="diag",
    n_iter=1000,
    random_state=42,
    verbose=False
)

Y = y.reshape(-1, 1)
hmm.fit(Y)

hidden_states = hmm.predict(Y)
state_probs = hmm.predict_proba(Y)

baseline_hmm_df = d_nlp.copy()
baseline_hmm_df["latent_state"] = hidden_states

# sort states by learned mean delta_vix so labels are interpretable
state_means = pd.Series(hmm.means_.flatten()).sort_values()
state_order = state_means.index.tolist()
state_label_map = {
    old_state: new_label
    for new_label, old_state in enumerate(state_order)
}

baseline_hmm_df["ordered_state"] = baseline_hmm_df["latent_state"].map(state_label_map)

print("Learned emission means:")
print(hmm.means_.flatten())

print("Learned transition matrix:")
print(hmm.transmat_)

baseline_hmm_df.groupby("ordered_state")[y_col].agg(["count", "mean", "std", "min", "max"])

Learned emission means:
[0.18551587 0.76584238 2.35001744]
Learned transition matrix:
[[0.35526144 0.55638942 0.08834913]
 [0.25791199 0.59265305 0.14943497]
 [0.03817973 0.27836773 0.68345255]]


,count,mean,std,min,max
ordered_state,,,,,
0,45,0.175111,0.094739,0.00,0.34
1,90,0.830444,0.328411,0.33,1.71
2,48,2.482292,1.333570,0.10,5.74


### Modeling set up

In [7]:
state_col = "ordered_state"

d_nlp = d_nlp.copy()
d_full = d_full.copy()

d_nlp[state_col] = baseline_hmm_df.loc[d_nlp.index, state_col].to_numpy()
d_full[state_col] = baseline_hmm_df.loc[d_full.index, state_col].to_numpy()

In [8]:
def make_transition_data(d, X, state_col="ordered_state"):
    """
    Converts model matrix X_t and inferred states Z_t into:
        predictors: [current_state_dummies, X_t]
        response: Z_{t+1}
    """

    d = d.copy()
    states = d[state_col].to_numpy().astype(int)

    X_t = X[:-1]
    state_t = states[:-1]
    y_next = states[1:]

    state_dummies = pd.get_dummies(
        state_t,
        prefix="current_state",
        drop_first=False
    )

    X_transition = np.column_stack([
        state_dummies.to_numpy(),
        X_t
    ])

    state_feature_names = state_dummies.columns.tolist()

    return X_transition, y_next, state_t, state_feature_names

In [9]:
X_trans_nlp, y_trans_nlp, state_t_nlp, state_feature_names = make_transition_data(
    d_nlp,
    X_nlp,
    state_col=state_col
)

X_trans_full, y_trans_full, state_t_full, state_feature_names_full = make_transition_data(
    d_full,
    X_full,
    state_col=state_col
)

feature_names_nlp = state_feature_names + X_nlp_cols
feature_names_full = state_feature_names_full + X_full_cols

print("NLP transition data:", X_trans_nlp.shape, y_trans_nlp.shape)
print("Full transition data:", X_trans_full.shape, y_trans_full.shape)
print("NLP features:", feature_names_nlp)

NLP transition data: (182, 14) (182,)
Full transition data: (182, 20) (182,)
NLP features: ['current_state_0', 'current_state_1', 'current_state_2', 'nmf_topic_0_score', 'nmf_topic_1_score', 'nmf_topic_2_score', 'nmf_topic_3_score', 'nmf_topic_4_score', 'nmf_topic_5_score', 'nmf_topic_6_score', 'nmf_topic_7_score', 'nmf_topic_8_score', 'nmf_topic_9_score', 'nmf_topic_10_score']


Helper Function

In [10]:
def fit_bayesian_transition_model(
    X_transition,
    y_transition,
    feature_names,
    draws=2000,
    tune=1000,
    cores=1,
    target_accept=0.9,
    random_seed=42
):
    """
    Bayesian multinomial logistic regression:
        P(Z_{t+1} | Z_t, X_t)

    This is the Bayesian extension of the logistic transition model.
    """

    K = len(np.unique(y_transition))

    coords = {
        "obs_id": np.arange(X_transition.shape[0]),
        "feature": feature_names,
        "class": [f"state_{i}" for i in range(K)]
    }

    with pm.Model(coords=coords) as model:
        X_data = pm.Data("X", X_transition, dims=("obs_id", "feature"))
        y_data = pm.Data("y", y_transition, dims="obs_id")

        alpha = pm.Normal("alpha", mu=0.0, sigma=1.5, dims="class")
        beta = pm.Normal("beta", mu=0.0, sigma=1.0, dims=("feature", "class"))

        logits = alpha + pt.dot(X_data, beta)

        p = pm.Deterministic(
            "p",
            pm.math.softmax(logits, axis=1),
            dims=("obs_id", "class")
        )

        pm.Categorical(
            "obs",
            p=p,
            observed=y_data,
            dims="obs_id"
        )

        trace = pm.sample(
            draws=draws,
            tune=tune,
            cores=cores,
            chains=2,
            target_accept=target_accept,
            random_seed=random_seed
        )

    return model, trace

### NLP-Only Model

In [11]:
model_nlp, trace_nlp = fit_bayesian_transition_model(
    X_trans_nlp,
    y_trans_nlp,
    feature_names_nlp,
    draws=500,
    tune=500,
    cores=1,
    target_accept=0.9,
    random_seed=42
)

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [alpha, beta]


Output()

Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 2426 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


### With Financial Controls

In [15]:
model_full, trace_full = fit_bayesian_transition_model(
    X_trans_full,
    y_trans_full,
    feature_names_full,
    draws=500,
    tune=500,
    cores=4
)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha, beta]


Output()

Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 1622 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


## Interpretation

In [12]:
az.summary(trace_nlp, var_names=["alpha", "beta"], hdi_prob=0.90)

,mean,sd,hdi_5%,hdi_95%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
alpha[state_0],-0.333,0.982,-1.774,1.419,0.023,0.036,1821.0,614.0,1.00
alpha[state_1],0.552,1.001,-1.089,2.208,0.023,0.031,1830.0,830.0,1.00
alpha[state_2],-0.271,0.954,-1.811,1.210,0.022,0.030,1838.0,821.0,1.00
"beta[current_state_0, state_0]",0.813,0.807,-0.478,2.128,0.017,0.028,2244.0,745.0,1.01
"beta[current_state_0, state_1]",0.225,0.732,-0.883,1.476,0.017,0.025,1865.0,733.0,1.01
"beta[current_state_0, state_2]",-1.016,0.751,-2.288,0.234,0.020,0.024,1463.0,643.0,1.00
"beta[current_state_1, state_0]",0.355,0.758,-0.895,1.572,0.014,0.031,2778.0,710.0,1.00
"beta[current_state_1, state_1]",0.414,0.750,-0.792,1.653,0.014,0.022,2689.0,854.0,1.00
"beta[current_state_1, state_2]",-0.735,0.742,-1.977,0.416,0.014,0.026,2987.0,895.0,1.00
"beta[current_state_2, state_0]",-1.315,0.760,-2.489,-0.026,0.018,0.023,1815.0,847.0,1.00


In [17]:
pd.set_option('display.max_rows', None)
az.summary(trace_full, var_names=["alpha", "beta"], hdi_prob=0.90)

,mean,sd,hdi_5%,hdi_95%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
alpha[state_0],-0.394,1.045,-2.000,1.455,0.024,0.034,1873.0,641.0,1.00
alpha[state_1],0.762,0.990,-0.907,2.324,0.024,0.034,1550.0,767.0,1.00
alpha[state_2],-0.298,0.993,-1.907,1.310,0.023,0.031,1862.0,731.0,1.00
"beta[current_state_0, state_0]",0.647,0.790,-0.721,1.897,0.016,0.027,2418.0,706.0,1.00
"beta[current_state_0, state_1]",0.156,0.728,-1.088,1.276,0.016,0.024,2243.0,898.0,1.01
"beta[current_state_0, state_2]",-0.828,0.807,-2.147,0.480,0.019,0.027,1815.0,791.0,1.00
"beta[current_state_1, state_0]",0.288,0.770,-1.018,1.502,0.016,0.025,2361.0,841.0,1.01
"beta[current_state_1, state_1]",0.421,0.733,-0.776,1.615,0.015,0.027,2263.0,698.0,1.01
"beta[current_state_1, state_2]",-0.708,0.758,-1.954,0.493,0.016,0.028,2327.0,755.0,1.00
"beta[current_state_2, state_0]",-1.151,0.847,-2.651,0.101,0.016,0.029,2869.0,790.0,1.00


In [21]:
def summarize_high_state_effects(
    trace,
    feature_names,
    target_state_index=2,
    hdi_prob=0.90
):
    target_state_name = f"state_{target_state_index}"

    beta_samples = trace.posterior["beta"]
    high_beta = beta_samples.sel({"class": target_state_name})

    summary = az.summary(high_beta, hdi_prob=hdi_prob).reset_index()
    summary["feature"] = feature_names

    prob_positive = (
        high_beta
        .stack(sample=("chain", "draw"))
        .values > 0
    ).mean(axis=1)

    summary["prob_positive"] = prob_positive

    return summary[
        ["feature", "mean", "sd", "hdi_5%", "hdi_95%", "prob_positive"]
    ].sort_values("mean", ascending=False)

In [22]:
summary_high_nlp = summarize_high_state_effects(
    trace_nlp,
    feature_names_nlp,
    target_state_index=2
)

summary_high_full = summarize_high_state_effects(
    trace_full,
    feature_names_full,
    target_state_index=2
)

summary_high_nlp

,feature,mean,sd,hdi_5%,hdi_95%,prob_positive
2,current_state_2,1.692,0.789,0.484,3.069,0.981
12,nmf_topic_9_score,0.229,0.603,-0.788,1.166,0.657
8,nmf_topic_5_score,0.172,0.630,-0.781,1.253,0.618
10,nmf_topic_7_score,0.171,0.615,-0.749,1.327,0.596
7,nmf_topic_4_score,0.161,0.644,-0.946,1.130,0.613
3,nmf_topic_0_score,0.153,0.593,-0.794,1.099,0.603
4,nmf_topic_1_score,0.085,0.591,-0.808,1.128,0.559
5,nmf_topic_2_score,0.083,0.585,-0.931,0.968,0.568
13,nmf_topic_10_score,0.032,0.618,-0.912,1.036,0.513
9,nmf_topic_6_score,0.008,0.655,-0.988,1.141,0.516


In [26]:
def posterior_transition_prob_for_scenario(
    trace,
    feature_names,
    base_x,
    target_state_index=2
):
    """
    Computes posterior samples of P(next_state = target_state_index)
    for a standardized feature vector base_x.
    """

    beta = trace.posterior["beta"].stack(sample=("chain", "draw"))
    alpha = trace.posterior["alpha"].stack(sample=("chain", "draw"))

    x = np.asarray(base_x)

    logits = alpha.values + np.einsum("f,fcs->cs", x, beta.values)

    logits = logits - logits.max(axis=0, keepdims=True)
    probs = np.exp(logits) / np.exp(logits).sum(axis=0, keepdims=True)

    return probs[target_state_index, :]

def summarize_probability_effects(
    trace,
    feature_names,
    nlp_features,
    target_state_index=2,
    low_value=-1,
    high_value=1
):
    rows = []

    for feature in nlp_features:
        if feature not in feature_names:
            continue

        idx = feature_names.index(feature)

        low_x = np.zeros(len(feature_names))
        high_x = np.zeros(len(feature_names))

        low_x[idx] = low_value
        high_x[idx] = high_value

        p_low = posterior_transition_prob_for_scenario(
            trace,
            feature_names,
            low_x,
            target_state_index=target_state_index
        )

        p_high = posterior_transition_prob_for_scenario(
            trace,
            feature_names,
            high_x,
            target_state_index=target_state_index
        )

        diff = p_high - p_low

        rows.append({
            "feature": feature,
            "mean_prob_change": diff.mean(),
            "hdi_5": np.quantile(diff, 0.05),
            "hdi_95": np.quantile(diff, 0.95),
            "prob_positive": (diff > 0).mean()
        })

    return pd.DataFrame(rows).sort_values(
        "mean_prob_change",
        ascending=False
    )

In [27]:
prob_effects_nlp = summarize_probability_effects(
    trace_nlp,
    feature_names_nlp,
    X_nlp_cols,
    target_state_index=2
)

prob_effects_full = summarize_probability_effects(
    trace_full,
    feature_names_full,
    X_nlp_cols,
    target_state_index=2
)

prob_effects_nlp

,feature,mean_prob_change,hdi_5,hdi_95,prob_positive
9,nmf_topic_9_score,0.099428,-0.054644,0.284865,0.854
0,nmf_topic_0_score,0.096912,-0.068084,0.286699,0.829
7,nmf_topic_7_score,0.081752,-0.075491,0.283543,0.806
4,nmf_topic_4_score,0.076652,-0.087856,0.251102,0.794
5,nmf_topic_5_score,0.075776,-0.086299,0.261140,0.775
1,nmf_topic_1_score,0.048141,-0.111643,0.223250,0.687
2,nmf_topic_2_score,0.031430,-0.143393,0.202737,0.623
10,nmf_topic_10_score,0.017535,-0.157735,0.189124,0.600
6,nmf_topic_6_score,0.017226,-0.187675,0.214028,0.565
3,nmf_topic_3_score,-0.007681,-0.203410,0.171138,0.498


In [28]:
prob_effect_comparison = prob_effects_nlp.merge(
    prob_effects_full,
    on="feature",
    suffixes=("_nlp_only", "_with_controls")
)

prob_effect_comparison

,feature,mean_prob_change_nlp_only,hdi_5_nlp_only,hdi_95_nlp_only,prob_positive_nlp_only,mean_prob_change_with_controls,hdi_5_with_controls,hdi_95_with_controls,prob_positive_with_controls
0,nmf_topic_9_score,0.099428,-0.054644,0.284865,0.854,-0.009044,-0.208704,0.179977,0.468
1,nmf_topic_0_score,0.096912,-0.068084,0.286699,0.829,0.013875,-0.178942,0.212912,0.555
2,nmf_topic_7_score,0.081752,-0.075491,0.283543,0.806,-0.006322,-0.197527,0.193459,0.476
3,nmf_topic_4_score,0.076652,-0.087856,0.251102,0.794,-0.029069,-0.215407,0.127955,0.396
4,nmf_topic_5_score,0.075776,-0.086299,0.261140,0.775,-0.011609,-0.197277,0.168110,0.451
5,nmf_topic_1_score,0.048141,-0.111643,0.223250,0.687,-0.128888,-0.358487,0.067918,0.155
6,nmf_topic_2_score,0.031430,-0.143393,0.202737,0.623,-0.033090,-0.223435,0.140651,0.375
7,nmf_topic_10_score,0.017535,-0.157735,0.189124,0.600,-0.055012,-0.247882,0.123993,0.308
8,nmf_topic_6_score,0.017226,-0.187675,0.214028,0.565,-0.007223,-0.198335,0.185645,0.476
9,nmf_topic_3_score,-0.007681,-0.203410,0.171138,0.498,-0.031538,-0.260392,0.162214,0.420


In [29]:
prob_effect_comparison.to_csv("prob_effect_comparison.csv")

---

In [22]:
topic_mapping = {}
topics = [
    'Economy, taxes and tariffs', 
    'Border security and immigration', 
    'Foreign policy', 
    'Energy', 
    'Environment', 
    'Government reform', 
    'Legal battles', 
    'Media attacks', 
    'Political opposition attacks', 
    'Patriotism', 
    'Domestic and social initiatives'
    ]

topic_cols = [c for c in baseline_hmm_df.columns if c.startswith("nmf_topic") and c.endswith("_score")]
for c, t in zip(topic_cols, topics):
    topic_mapping[c] = t


def map_topics_from_columns(col: pd.Series):
    new_cols = []
    for val in col:
        new_cols.append(topic_mapping.get(val, val))
    
    return new_cols

In [23]:
topic_mapping

{'nmf_topic_0_score': 'Economy, taxes and tariffs',
 'nmf_topic_1_score': 'Border security and immigration',
 'nmf_topic_2_score': 'Foreign policy',
 'nmf_topic_3_score': 'Energy',
 'nmf_topic_4_score': 'Environment',
 'nmf_topic_5_score': 'Government reform',
 'nmf_topic_6_score': 'Legal battles',
 'nmf_topic_7_score': 'Media attacks',
 'nmf_topic_8_score': 'Political opposition attacks',
 'nmf_topic_9_score': 'Patriotism',
 'nmf_topic_10_score': 'Domestic and social initiatives'}

---